
Programación para Análisis de Datos

Integrantes: JAVIER HERNAN MARTINEZ CEBALLOS - JHONNY ARLEY TOBON ACEVEDO

Docente: ING. ANDRES FELIPE PALACIO

PREICA2602B020048

IU Digital Antioquia

Noviembre-2026

In [ ]:
import json
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

print("Entorno configurado correctamente.")

Mounted at /content/drive
✓ Entorno configurado correctamente.


In [ ]:
# Definir el sitio web que se va a analizar.
url = 'https://www.calzatodo.com.co/es/hombres/tenis'

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers, timeout=20)
response.raise_for_status()
soup = BeautifulSoup(response.content, "html.parser")
print("Petición exitosa")

Petición exitosa


In [ ]:
zapatos_lista = []
scripts_json_ld = soup.find_all('script', type='application/ld+json')

for script in scripts_json_ld:
    try:
        datos = json.loads(script.string or script.get_text())
    except (json.JSONDecodeError, TypeError):
        continue

    if not isinstance(datos, dict):
        continue

    tipo = datos.get('@type')
    if tipo == 'ItemList':
        productos = [
            wrapper.get('item', {})
            for wrapper in datos.get('itemListElement', [])
            if isinstance(wrapper, dict)
        ]
    elif tipo == 'Product':
        productos = [datos]
    else:
        continue

    for producto in productos:
        if not isinstance(producto, dict) or not producto:
            continue

        marca = producto.get('brand', 'Calzatodo')
        if isinstance(marca, dict):
            marca = marca.get('name', 'Calzatodo')

        ofertas = producto.get('offers', {})
        if isinstance(ofertas, list):
            ofertas = ofertas[0] if ofertas else {}
        precio_num = None
        if isinstance(ofertas, dict):
            precio_num = (
                ofertas.get('highPrice')
                or ofertas.get('lowPrice')
                or ofertas.get('price')
            )
        precio = f"${float(precio_num):,.2f}" if precio_num else 'No disponible'

        zapatos_lista.append({
            "Referencia (SKU)": producto.get('sku', producto.get('productID', 'No disponible')),
            "Nombre/Marca": f"[{marca}] {producto.get('name', 'No disponible')}",
            "Valores": precio,
            "Características": "Calzado outdoor, Deportivo, Hombre",
            "Color": producto.get('color', 'No disponible'),
            "Categoría": producto.get('category', 'No disponible'),
            "Material": producto.get('material', 'No disponible'),
        })

if not zapatos_lista:
    for prod in soup.select('[class*="product"], [class*="summary"]'):
        link = prod.find('a', href=True)
        nombre = link.get_text(strip=True) if link else ''
        if nombre and len(nombre) > 5:
            zapatos_lista.append({
                "Referencia (SKU)": 'No disponible',
                "Nombre/Marca": f"[Calzatodo] {nombre}",
                "Valores": "Consultar precio dinámico",
                "Características": "Calzado outdoor, Deportivo, Hombre",
                "Color": "No disponible",
                "Categoría": "No disponible",
                "Material": "No disponible",
            })

columnas_producto = [
    "Referencia (SKU)", "Nombre/Marca", "Valores", "Características",
    "Color", "Categoría", "Material",
]
df_producto = pd.DataFrame(zapatos_lista, columns=columnas_producto)
if not df_producto.empty:
    df_producto = df_producto.drop_duplicates(subset=['Nombre/Marca'])

{
    "@context": "https://schema.org/",
    "@type": "Product",
    "@id": "https://www.calzatodo.com.co/tenis-118155bbk-28121000211/p",
    "name": "Tenis Deportivo de Hombre Skechers  118155bbk",
    "brand": {
        "@type": "Brand",
        "name": "Skechers"
    },
    "image": "https://calzatodocol.vtexassets.com/arquivos/ids/496284/28121000211_negro_01.jpg.jpg?v=639291998536900000",
    "description": "Tenis de Hombre marca Skechers",
    "mpn": "28121000211_461847",
    "sku": "461847",
    "offers": {
        "@type": "AggregateOffer",
        "lowPrice": 249900,
        "highPrice": 249900,
        "priceCurrency": "COP",
        "offers": [
            {
                "@type": "Offer",
                "price": 249900,
                "priceCurrency": "COP",
                "availability": "http://schema.org/InStock",
                "sku": "461847",
                "itemCondition": "http://schema.org/NewCondition",
                "priceValidUntil": "2027-11-11T00:48:40

Web Scraping con paginas - utilizando Beautifulosup

In [ ]:
# Lista global para consolidar los productos de todas las páginas
todos_los_zapatos = []

paginas_a_descargar = 5
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

print("Iniciando la extracción con paginación...")

for pagina in range(1, paginas_a_descargar + 1):
    url_dinamica = f"https://www.calzatodo.com.co/es/hombres?page={pagina}"
    print(f"Procesando página {pagina} de {paginas_a_descargar}... ", end="")

    try:
        response = requests.get(url_dinamica, headers=headers, timeout=20)
        response.raise_for_status()
        soup_pag = BeautifulSoup(response.content, 'html.parser')
        scripts_json_ld = soup_pag.find_all('script', type='application/ld+json')

        conteo_pagina = 0
        for script in scripts_json_ld:
            try:
                datos = json.loads(script.string or script.get_text())
            except (json.JSONDecodeError, TypeError):
                continue

            if not isinstance(datos, dict):
                continue

            if datos.get('@type') == 'ItemList':
                elementos = datos.get('itemListElement', [])
            else:
                continue

            for elemento in elementos:
                if not isinstance(elemento, dict):
                    continue
                producto = elemento.get('item', {})
                if not isinstance(producto, dict) or not producto:
                    continue

                marca = producto.get('brand', 'Calzatodo')
                if isinstance(marca, dict):
                    marca = marca.get('name', 'Calzatodo')

                ofertas = producto.get('offers', {})
                if isinstance(ofertas, list):
                    ofertas = ofertas[0] if ofertas else {}
                precio_num = None
                if isinstance(ofertas, dict):
                    precio_num = (
                        ofertas.get('highPrice')
                        or ofertas.get('lowPrice')
                        or ofertas.get('price')
                    )
                precio = f"${float(precio_num):,.2f}" if precio_num else 'No disponible'

                todos_los_zapatos.append({
                    "Referencia (SKU)": producto.get('sku', producto.get('productID', 'No disponible')),
                    "Nombre/Marca": f"[{marca}] {producto.get('name', 'No disponible')}",
                    "Valor": precio,
                    "Características": f"Calzado Outdoor, Deportivo, Hombre (Pág {pagina})",
                })
                conteo_pagina += 1

        print(f"Completada: {conteo_pagina} productos.")
        time.sleep(1.5)

    except (requests.RequestException, ValueError, TypeError) as error:
        print(f"Error al procesar la página {pagina}: {error}")

# Crear el DataFrame consolidador
df_paginado = pd.DataFrame(
    todos_los_zapatos,
    columns=["Referencia (SKU)", "Nombre/Marca", "Valor", "Características"],
)
if not df_paginado.empty:
    df_paginado = df_paginado.drop_duplicates(subset=['Nombre/Marca'])

print(f"\nExtracción finalizada. Total de productos únicos: {len(df_paginado)}")
if not df_paginado.empty:
    display(df_paginado)

nombre_archivo_total = "productos_calzatodo_todas_las_paginas.csv"
df_paginado.to_csv(nombre_archivo_total, index=False, encoding='utf-8-sig', sep=';')
print(f"CSV generado: {nombre_archivo_total}")

Iniciando la extracción multicanal con paginación...
Procesando página 1 de 5... ¡Completada! Se encontraron 20 productos.
Procesando página 2 de 5... ¡Completada! Se encontraron 20 productos.
Procesando página 3 de 5... ¡Completada! Se encontraron 20 productos.
Procesando página 4 de 5... ¡Completada! Se encontraron 20 productos.
Procesando página 5 de 5... ¡Completada! Se encontraron 20 productos.

Extracción finalizada. Total de productos únicos recolectados: 80


,Referencia (SKU),Nombre/Marca,Valor,Características
0,461847,[Skechers] Tenis Deportivo de Hombre Skechers ...,"$249,900.00","Calzado Outdoor, Deportivo, Hombre (Pág 1)"
1,461868,[Skechers] Tenis Deportivo de Hombre Skechers ...,"$229,900.00","Calzado Outdoor, Deportivo, Hombre (Pág 1)"
2,461861,[Skechers] Tenis Deportivo de Hombre Skechers ...,"$299,900.00","Calzado Outdoor, Deportivo, Hombre (Pág 1)"
3,461882,[Skechers] Zapato Deportivo de Hombre Skechers...,"$299,900.00","Calzado Outdoor, Deportivo, Hombre (Pág 1)"
4,465521,[24 Walks] Tenis Institucional de Hombre 24 wa...,"$111,900.00","Calzado Outdoor, Deportivo, Hombre (Pág 1)"
5,454448,[Rider] Sandalia Beach de Hombre Rider 12675c ...,"$35,900.00","Calzado Outdoor, Deportivo, Hombre (Pág 1)"
6,454443,[Rider] Sandalia Beach de Hombre Rider 12675c ...,"$35,900.00","Calzado Outdoor, Deportivo, Hombre (Pág 1)"
7,454433,[Cartago] Sandalia Beach de Hombre Cartago 122...,"$69,900.00","Calzado Outdoor, Deportivo, Hombre (Pág 1)"
8,462666,[24 Walks] Zapato Casual de Hombre 24 walks Te...,"$129,900.00","Calzado Outdoor, Deportivo, Hombre (Pág 1)"
11,444597,[Skechers] Tenis Deportivo de Hombre Skechers ...,"$329,900.00","Calzado Outdoor, Deportivo, Hombre (Pág 1)"
